In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


True
NVIDIA GeForce RTX 4070 Ti


In [3]:
cd ice_sleep_detpj/

/home/jovyan/ice_sleep_detpj/ice_sleep_detpj


In [47]:
cd data/

/home/jovyan/ice_sleep_detpj/data


In [55]:
!mkdir -p all_images_준통제
!find "ice_control_dataset/ice_project/Validation/[원천]keypoint_준통제환경" -name "*.jpg" -exec cp {} all_images_준통제/ \;

In [4]:
cd ..

/home/jovyan/ice_sleep_detpj


In [2]:
ls

 CITATION.cff              classify/                 requirements.txt
 CONTRIBUTING.md           copy_txt_to_yolo.py       runs/
 LICENSE                   data/                     segment/
 README.md                 detect.py                 train.ipynb
 README.zh-CN.md           export.py                 train.py
'Untitled Folder'/         feature_scaler.pkl        train_lstm.ipynb
 __pycache__/              feature_scaler_12_6.pkl   tutorial.ipynb
 benchmarks.py             hubconf.py                utils/
 best_lstm_model.h5        ice_sleep_detpj/          val.py
 best_lstm_model_12_6.h5   models/                   yolov5s.pt
 best_lstm_model_v2.h5     pyproject.toml


In [8]:
!find . -name "*ice_driver.yaml"

./data/ice_driver.yaml


In [3]:
!find . -name "*train.py"

./ice_sleep_detpj/utils/train.py
./ice_sleep_detpj/segment/train.py
./ice_sleep_detpj/classify/train.py
./ice_sleep_detpj/train.py


In [46]:
!unzip -q /home/jovyan/ice_sleep_detpj/data/ice_control_dataset/ice_project/Validation/[원천]keypoint_준통제환경.zip -d /home/jovyan/ice_sleep_detpj/data/ice_control_dataset/ice_project/Validation/[원천]keypoint_준통제환경

In [1]:
!pip install opencv-python

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 112.3 MB/s eta 0:00:0000:0100:01


In [50]:
!rm -rf /home/jovyan/ice_sleep_detpj/data/all_images_준통제


In [61]:
import json
import os

def convert_bbox(bbox, img_width, img_height):
    x_min, y_min, x_max, y_max = map(float, bbox)
    x_center = (x_min + x_max) / 2 / img_width
    y_center = (y_min + y_max) / 2 / img_height
    w = (x_max - x_min) / img_width
    h = (y_max - y_min) / img_height
    return x_center, y_center, w, h

def merge_bboxes(b1, b2):
    x_min = min(float(b1[0]), float(b2[0]))
    y_min = min(float(b1[1]), float(b2[1]))
    x_max = max(float(b1[2]), float(b2[2]))
    y_max = max(float(b1[3]), float(b2[3]))
    return [x_min, y_min, x_max, y_max]

def json_to_yolo(json_path, output_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    width = int(data['FileInfo']['Width'])
    height = int(data['FileInfo']['Height'])
    bboxes = data['ObjectInfo']['BoundingBox']

    result_lines = []

    # 👁️ 눈 (양쪽 눈 기준 합쳐서)
    leye = bboxes.get('Leye')
    reye = bboxes.get('Reye')
    if leye['isVisible'] and reye['isVisible']:
        is_open = leye['Opened'] or reye['Opened']
        class_id = 0 if is_open else 1  # 0: eye_open, 1: eye_closed
        merged_box = merge_bboxes(leye['Position'], reye['Position'])
        coords = convert_bbox(merged_box, width, height)
        result_lines.append(f"{class_id} {' '.join(map(str, coords))}")

    # 👄 입
    mouth = bboxes.get('Mouth')
    if mouth['isVisible']:
        class_id = 2 if mouth['Opened'] else 3  # 2: mouth_open, 3: mouth_closed
        coords = convert_bbox(mouth['Position'], width, height)
        result_lines.append(f"{class_id} {' '.join(map(str, coords))}")

    # 저장
    base_name = os.path.splitext(os.path.basename(json_path))[0]
    output_file = os.path.join(output_path, base_name + '.txt')
    with open(output_file, 'w') as f:
        f.write('\n'.join(result_lines))

# 📁 Jupyter 환경 기준 경로 (수정됨)
input_dir = 'data/ice_th/YOLO_dataset/labels/val'        # JSON 파일들
output_dir = 'data/ice_th/YOLO_dataset/labels/val_txt'   # YOLO txt 저장 경로

os.makedirs(output_dir, exist_ok=True)

# 변환 실행
for filename in os.listdir(input_dir):
    if filename.endswith('.json'):
        json_path = os.path.join(input_dir, filename)
        json_to_yolo(json_path, output_dir)

print("✅ 변환 완료!")


✅ 변환 완료!


In [ ]:
import os
import numpy as np

def generate_y_true_from_folders(controlled_root, realroad_root, save_path="data/y_true_labeling.npy"):
    """
    통제환경은 졸음(1), 실제 도로 환경은 정상(0)으로 라벨링.
    폴더명(사람 단위)을 기준으로 라벨 생성 후 numpy 배열로 저장.

    Args:
        controlled_root (str): 통제환경 데이터 루트 폴더 경로
        realroad_root (str): 실제 도로 환경 데이터 루트 폴더 경로
        save_path (str): 저장할 y_true 경로 (npy 파일)
    """
    y_true = []
    person_ids = []

    # ✅ 통제환경: 졸음 = 1
    if os.path.exists(controlled_root):
        for person_folder in sorted(os.listdir(controlled_root)):
            if os.path.isdir(os.path.join(controlled_root, person_folder)):
                person_ids.append(person_folder)
                y_true.append(1)

    # ✅ 실제 도로 환경: 정상 = 0
    if os.path.exists(realroad_root):
        for person_folder in sorted(os.listdir(realroad_root)):
            if os.path.isdir(os.path.join(realroad_root, person_folder)):
                person_ids.append(person_folder)
                y_true.append(0)

    y_true = np.array(y_true)
    np.save(save_path, y_true)

    print(f"총 {len(y_true)}명에 대해 y_true 저장 완료 ✅")
    print(f"졸음: {np.sum(y_true)}, 정상: {len(y_true) - np.sum(y_true)}")
    print(f"저장 경로: {save_path}")
    return y_true, person_ids

# ✅ 실행
controlled_root = "data/ice_control_dataset/ice_project/Training/[라벨]bbox_통제환경"
realroad_root = "data/ice_control_dataset/ice_project/Training/[라벨]bbox_실제도로환경/2.승용"


y_true, person_ids = generate_y_true_from_folders(controlled_root, realroad_root)


In [5]:
import json, os
import numpy as np

# 📂 JSON 파일 경로 (YOLO 학습 시 사용한 라벨 기준)
json_dir = "data/ice_lstm_train_passanger/YOLO_dataset/labels/train"

# 📂 저장할 경로
save_dir = "data/ice_lstm_seq"
os.makedirs(save_dir, exist_ok=True)

# 📌 Feature 추출 함수
def extract_feature(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)

    bb = data["ObjectInfo"]["BoundingBox"]
    def area(pos):
        return (float(pos[2]) - float(pos[0])) * (float(pos[3]) - float(pos[1]))

    return [
        int(bb["Leye"]["Opened"]),
        int(bb["Reye"]["Opened"]),
        int(bb["Mouth"]["Opened"]),
        area(bb["Face"]["Position"]),
        area(bb["Leye"]["Position"]),
        area(bb["Reye"]["Position"]),
        area(bb["Mouth"]["Position"]),
    ]

# ✅ feature vector 리스트 생성
json_files = sorted([f for f in os.listdir(json_dir) if f.endswith('.json')])
features = [extract_feature(os.path.join(json_dir, f)) for f in json_files]

# ✅ 시퀀스 데이터 구성
X_seq, y_seq = [], []
seq_len = 30

for i in range(len(features) - seq_len):
    seq = features[i:i+seq_len]
    closed_count = sum((1 - f[0]) + (1 - f[1]) for f in seq)
    label = 1 if closed_count > 15 else 0
    X_seq.append(seq)
    y_seq.append(label)

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

# ✅ 저장
np.save(os.path.join(save_dir, "X_seq.npy"), X_seq)
np.save(os.path.join(save_dir, "y_seq.npy"), y_seq)

# ✅ 요약 출력
print("✅ LSTM 학습용 시퀀스 데이터 생성 및 저장 완료!")
print(f"X_seq shape: {X_seq.shape}")
print(f"y_seq shape: {y_seq.shape}")
print(f"Positive(졸음) 샘플 수: {np.sum(y_seq)} / Negative(정상): {len(y_seq) - np.sum(y_seq)}")
print(f"📁 저장 경로: {save_dir}/X_seq.npy, y_seq.npy")


✅ LSTM 학습용 시퀀스 데이터 생성 및 저장 완료!
X_seq shape: (70914, 30, 7)
y_seq shape: (70914,)
Positive(졸음) 샘플 수: 6926 / Negative(정상): 63988
📁 저장 경로: data/ice_lstm_seq/X_seq.npy, y_seq.npy


In [ ]:
# 통제 포함을 위한 승용데이터 seq 형성.
import os
import json
import numpy as np

# ✅ Feature 추출 함수
def extract_feature(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)

    bb = data["ObjectInfo"]["BoundingBox"]

    def safe_area(pos):
        try:
            return max(0, (float(pos[2]) - float(pos[0])) * (float(pos[3]) - float(pos[1])))
        except:
            return 0.0

    left_eye_open = int(bb["Leye"]["Opened"])
    right_eye_open = int(bb["Reye"]["Opened"])
    mouth_open = int(bb["Mouth"]["Opened"])

    face_area = safe_area(bb["Face"]["Position"])
    left_eye_area = safe_area(bb["Leye"]["Position"])
    right_eye_area = safe_area(bb["Reye"]["Position"])
    mouth_area = safe_area(bb["Mouth"]["Position"])

    if face_area > 0:
        left_eye_ratio = left_eye_area / face_area
        right_eye_ratio = right_eye_area / face_area
        mouth_ratio = mouth_area / face_area
        face_area_log = np.log(face_area + 1)
    else:
        left_eye_ratio = right_eye_ratio = mouth_ratio = face_area_log = 0.0

    return [
        left_eye_open,
        right_eye_open,
        mouth_open,
        face_area_log,
        left_eye_ratio,
        right_eye_ratio,
        mouth_ratio,
    ]

# ✅ 시퀀스 생성 함수
def create_sequences(features, seq_len=30):
    X_seq, y_seq = [], []
    for i in range(len(features) - seq_len):
        seq = features[i:i + seq_len]

        both_eyes_closed_count = sum(1 for f in seq if f[0] == 0 and f[1] == 0)
        max_consecutive_closed = 0
        current_consecutive = 0
        for f in seq:
            if f[0] == 0 and f[1] == 0:
                current_consecutive += 1
                max_consecutive_closed = max(max_consecutive_closed, current_consecutive)
            else:
                current_consecutive = 0

        is_drowsy = (
            both_eyes_closed_count >= 9 or  
            max_consecutive_closed >= 6    
        )

        label = 1 if is_drowsy else 0
        X_seq.append(seq)
        y_seq.append(label)

    return np.array(X_seq), np.array(y_seq)

# ✅ 전체 경로 탐색 및 처리 함수
def process_and_create_seq(json_dir, name="train"):
    all_features = []
    json_count = 0
    for root, _, files in os.walk(json_dir):  # 하위 폴더까지 탐색
        for jf in sorted(files):
            if jf.endswith('.json'):
                try:
                    fpath = os.path.join(root, jf)
                    all_features.append(extract_feature(fpath))
                    json_count += 1
                except Exception as e:
                    print(f"⚠️ 오류: {jf} - {e}")
    print(f"📂 {name} JSON 개수: {json_count}")
    return create_sequences(all_features)

# ✅ 경로 설정 (수정 가능)
train_json_root = "data/ice_control_dataset/ice_project/Training/[라벨]bbox_실제도로환경/2.승용"
val_json_root = "data/ice_control_dataset/ice_project/Validation/[라벨]bbox_실제도로환경/2.승용"
save_dir = "data/ice_lstm_seq_" \
""
os.makedirs(save_dir, exist_ok=True)

# ✅ 시퀀스 생성
X_train, y_train = process_and_create_seq(train_json_root, name="train")
X_val, y_val = process_and_create_seq(val_json_root, name="val")

# ✅ 저장
np.save(os.path.join(save_dir, "X_seq.npy"), X_train)
np.save(os.path.join(save_dir, "y_seq.npy"), y_train)
np.save(os.path.join(save_dir, "y_true.npy"), y_train)  # <- 추가
np.save(os.path.join(save_dir, "X_val_seq.npy"), X_val)
np.save(os.path.join(save_dir, "y_val_seq.npy"), y_val)
np.save(os.path.join(save_dir, "y_val_true.npy"), y_val) 

# ✅ 요약 출력
print("\n✅ 시퀀스 저장 완료!")
print(f"훈련 데이터: X={X_train.shape}, y={y_train.shape} (졸음: {np.sum(y_train)}, 정상: {len(y_train)-np.sum(y_train)})")
print(f"검증 데이터: X={X_val.shape}, y={y_val.shape} (졸음: {np.sum(y_val)}, 정상: {len(y_val)-np.sum(y_val)})")
# 50144

📂 train JSON 개수: 70945
📂 val JSON 개수: 8903

✅ 시퀀스 저장 완료!
훈련 데이터: X=(70915, 30, 7), y=(70915,) (졸음: 4514, 정상: 66401)
검증 데이터: X=(8873, 30, 7), y=(8873,) (졸음: 417, 정상: 8456)


In [ ]:
# 통제 환경 seq 형성하기.
import os
import json
import numpy as np

# ✅ Feature 추출 함수
def extract_feature(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)

    bb = data["ObjectInfo"]["BoundingBox"]

    def safe_area(pos):
        try:
            return max(0, (float(pos[2]) - float(pos[0])) * (float(pos[3]) - float(pos[1])))
        except:
            return 0.0

    left_eye_open = int(bb["Leye"]["Opened"])
    right_eye_open = int(bb["Reye"]["Opened"])
    mouth_open = int(bb["Mouth"]["Opened"])

    face_area = safe_area(bb["Face"]["Position"])
    left_eye_area = safe_area(bb["Leye"]["Position"])
    right_eye_area = safe_area(bb["Reye"]["Position"])
    mouth_area = safe_area(bb["Mouth"]["Position"])

    if face_area > 0:
        left_eye_ratio = left_eye_area / face_area
        right_eye_ratio = right_eye_area / face_area
        mouth_ratio = mouth_area / face_area
        face_area_log = np.log(face_area + 1)
    else:
        left_eye_ratio = right_eye_ratio = mouth_ratio = face_area_log = 0.0

    return [
        left_eye_open,
        right_eye_open,
        mouth_open,
        face_area_log,
        left_eye_ratio,
        right_eye_ratio,
        mouth_ratio,
    ]

# ✅ 시퀀스 생성 함수
def create_sequences(features, seq_len=30):
    X_seq, y_seq = [], []
    for i in range(len(features) - seq_len):
        seq = features[i:i + seq_len]

        both_eyes_closed_count = sum(1 for f in seq if f[0] == 0 and f[1] == 0)
        max_consecutive_closed = 0
        current_consecutive = 0
        for f in seq:
            if f[0] == 0 and f[1] == 0:
                current_consecutive += 1
                max_consecutive_closed = max(max_consecutive_closed, current_consecutive)
            else:
                current_consecutive = 0

        is_drowsy = (
            both_eyes_closed_count >= 9 or  # 약 0.4초
            max_consecutive_closed >= 6      # 약 0.3초
        )

        label = 1 if is_drowsy else 0
        X_seq.append(seq)
        y_seq.append(label)

    return np.array(X_seq), np.array(y_seq)

# ✅ 전체 경로 탐색 및 처리 함수
def process_and_create_seq(json_dir, name="train"):
    all_features = []
    json_count = 0
    for root, _, files in os.walk(json_dir):  # 하위 폴더까지 탐색
        for jf in sorted(files):
            if jf.endswith('.json'):
                try:
                    fpath = os.path.join(root, jf)
                    all_features.append(extract_feature(fpath))
                    json_count += 1
                except Exception as e:
                    print(f"⚠️ 오류: {jf} - {e}")
    print(f"📂 {name} JSON 개수: {json_count}")
    return create_sequences(all_features)

# ✅ 경로 설정 (수정 가능)
train_json_root = "data/ice_control_dataset/ice_project/Training/[라벨]bbox_통제환경"
val_json_root = "data/ice_control_dataset/ice_project/Validation/[라벨]bbox_통제환경"
save_dir = "data/ice_lstm_seq_통제전용"
os.makedirs(save_dir, exist_ok=True)

# ✅ 시퀀스 생성
X_train, y_train = process_and_create_seq(train_json_root, name="train")
X_val, y_val = process_and_create_seq(val_json_root, name="val")

# ✅ 저장
np.save(os.path.join(save_dir, "X_seq.npy"), X_train)
np.save(os.path.join(save_dir, "y_seq.npy"), y_train)
np.save(os.path.join(save_dir, "y_true.npy"), y_train)  # <- 추가
np.save(os.path.join(save_dir, "X_val_seq.npy"), X_val)
np.save(os.path.join(save_dir, "y_val_seq.npy"), y_val)
np.save(os.path.join(save_dir, "y_val_true.npy"), y_val)  # <- 추가

# ✅ 요약 출력
print("\n✅ 시퀀스 저장 완료!")
print(f"훈련 데이터: X={X_train.shape}, y={y_train.shape} (졸음: {np.sum(y_train)}, 정상: {len(y_train)-np.sum(y_train)})")
print(f"검증 데이터: X={X_val.shape}, y={y_val.shape} (졸음: {np.sum(y_val)}, 정상: {len(y_val)-np.sum(y_val)})")


📂 train JSON 개수: 100303
📂 val JSON 개수: 12563

✅ 시퀀스 저장 완료!
훈련 데이터: X=(100273, 30, 7), y=(100273,) (졸음: 48031, 정상: 52242)
검증 데이터: X=(12533, 30, 7), y=(12533,) (졸음: 6474, 정상: 6059)


In [ ]:
# seq combined 버전전
import numpy as np
import os

# ✅ 이미 각각 저장한 경우: 불러와서 합치기
# 승용
X_train_s = np.load("data/ice_lstm_seq_승용전용/X_seq.npy")
y_train_s = np.load("data/ice_lstm_seq_승용전용/y_seq.npy")
X_val_s = np.load("data/ice_lstm_seq_승용전용/X_val_seq.npy")
y_val_s = np.load("data/ice_lstm_seq_승용전용/y_val_seq.npy")

# 통제
X_train_c = np.load("data/ice_lstm_seq_통제전용/X_seq.npy")
y_train_c = np.load("data/ice_lstm_seq_통제전용/y_seq.npy")
X_val_c = np.load("data/ice_lstm_seq_통제전용/X_val_seq.npy")
y_val_c = np.load("data/ice_lstm_seq_통제전용/y_val_seq.npy")

# ✅ concat
X_train_combined = np.concatenate([X_train_s, X_train_c], axis=0)
y_train_combined = np.concatenate([y_train_s, y_train_c], axis=0)
X_val_combined = np.concatenate([X_val_s, X_val_c], axis=0)
y_val_combined = np.concatenate([y_val_s, y_val_c], axis=0)

# ✅ 저장
save_dir = "data/ice_lstm_seq_combined"
os.makedirs(save_dir, exist_ok=True)

np.save(os.path.join(save_dir, "X_seq.npy"), X_train_combined)
np.save(os.path.join(save_dir, "y_seq.npy"), y_train_combined)
np.save(os.path.join(save_dir, "X_val_seq.npy"), X_val_combined)
np.save(os.path.join(save_dir, "y_val_seq.npy"), y_val_combined)

# ✅ 요약 출력
print("\n✅ 통합 시퀀스 저장 완료!")
print(f"훈련 데이터: X={X_train_combined.shape}, y={y_train_combined.shape} (졸음: {np.sum(y_train_combined)}, 정상: {len(y_train_combined)-np.sum(y_train_combined)})")
print(f"검증 데이터: X={X_val_combined.shape}, y={y_val_combined.shape} (졸음: {np.sum(y_val_combined)}, 정상: {len(y_val_combined)-np.sum(y_val_combined)})")



✅ 통합 시퀀스 저장 완료!
훈련 데이터: X=(171188, 30, 7), y=(171188,) (졸음: 51092, 정상: 120096)
검증 데이터: X=(21406, 30, 7), y=(21406,) (졸음: 6692, 정상: 14714)


In [ ]:
import os
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.metrics import Precision, Recall

# ✅ 1. 시퀀스 데이터 불러오기
X_train = np.load("data/ice_lstm_seq_combined/X_seq.npy")
y_train = np.load("data/ice_lstm_seq_combined/y_seq.npy")
X_val = np.load("data/ice_lstm_seq_combined/X_val_seq.npy")
y_val = np.load("data/ice_lstm_seq_combined/y_val_seq.npy")

print(f"✅ 데이터 로드 완료:")
print(f"  📂 X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  📂 X_val:   {X_val.shape}, y_val:   {y_val.shape}")
print(f"  📊 클래스 분포 (Train) - 졸음: {np.sum(y_train)}, 정상: {len(y_train) - np.sum(y_train)}")

# ✅ 2. Feature Scaling
scaler = StandardScaler()

X_train_2d = X_train.reshape(-1, X_train.shape[-1])
X_val_2d = X_val.reshape(-1, X_val.shape[-1])

X_train_scaled = scaler.fit_transform(X_train_2d).reshape(X_train.shape)
X_val_scaled = scaler.transform(X_val_2d).reshape(X_val.shape)

joblib.dump(scaler, 'feature_scaler.pkl')
print("✅ Feature Scaling 및 저장 완료: feature_scaler.pkl")

# ✅ 3. 클래스 가중치 계산
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}
print(f"✅ 클래스 가중치: {class_weight_dict}")

# ✅ 4. LSTM 모델 정의
model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2]), dropout=0.2, recurrent_dropout=0.2),
    BatchNormalization(),

    LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2),
    BatchNormalization(),

    LSTM(32, dropout=0.2, recurrent_dropout=0.2),
    BatchNormalization(),

    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy', Precision(name='precision'), Recall(name='recall')]
)
model.summary()

# ✅ 5. 콜백 설정
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ModelCheckpoint("best_lstm_model.h5", monitor='val_loss', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1)
]

# ✅ 6. 학습 수행
history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=50,
    batch_size=32,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

# ✅ 7. 결과 출력
print("\n✅ 학습 완료!")
print(f"최종 Train Loss: {min(history.history['loss']):.4f}")
print(f"최종 Val Loss:   {min(history.history['val_loss']):.4f}")
print(f"최고 Val Accuracy: {max(history.history['val_accuracy']):.4f}")

2025-05-26 06:24:16.240653: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-26 06:24:16.260202: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-26 06:24:16.260223: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-26 06:24:16.260770: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-26 06:24:16.264229: I tensorflow/core/platform/cpu_feature_guar

✅ 데이터 로드 완료:
  📂 X_train: (171188, 30, 7), y_train: (171188,)
  📂 X_val:   (21406, 30, 7), y_val:   (21406,)
  📊 클래스 분포 (Train) - 졸음: 51092, 정상: 120096
✅ Feature Scaling 및 저장 완료: feature_scaler.pkl
✅ 클래스 가중치: {0: 0.7127131628030908, 1: 1.6752916307836843}


2025-05-26 06:24:17.551942: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-05-26 06:24:17.568043: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-05-26 06:24:17.568155: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 30, 128)           69632     
                                                                 
 batch_normalization (Batch  (None, 30, 128)           512       
 Normalization)                                                  
                                                                 
 lstm_1 (LSTM)               (None, 30, 64)            49408     
                                                                 
 batch_normalization_1 (Bat  (None, 30, 64)            256       
 chNormalization)                                                
                                                                 
 lstm_2 (LSTM)               (None, 32)                12416     
                                                                 
 batch_normalization_2 (Bat  (None, 32)                1

2025-05-26 06:24:22.195921: I external/local_xla/xla/service/service.cc:168] XLA service 0x7f4179222470 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-05-26 06:24:22.195954: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 Ti, Compute Capability 8.9
2025-05-26 06:24:22.200348: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-05-26 06:24:22.212754: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8902
I0000 00:00:1748240662.261185     391 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


5350/5350 [==============================] - ETA: 0s - loss: 0.0978 - accuracy: 0.9602 - precision: 0.9052 - recall: 0.9679
Epoch 1: val_loss improved from inf to 0.05485, saving model to best_lstm_model.h5
5350/5350 [==============================] - 409s 75ms/step - loss: 0.0978 - accuracy: 0.9602 - precision: 0.9052 - recall: 0.9679 - val_loss: 0.0548 - val_accuracy: 0.9752 - val_precision: 0.9308 - val_recall: 0.9946 - lr: 0.0010
Epoch 2/50
   2/5350 [..............................] - ETA: 6:10 - loss: 0.0042 - accuracy: 1.0000 - precision: 1.0000 - recall: 1.0000

/opt/conda/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


5350/5350 [==============================] - ETA: 0s - loss: 0.0504 - accuracy: 0.9802 - precision: 0.9524 - recall: 0.9826
Epoch 2: val_loss improved from 0.05485 to 0.04259, saving model to best_lstm_model.h5
5350/5350 [==============================] - 405s 76ms/step - loss: 0.0504 - accuracy: 0.9802 - precision: 0.9524 - recall: 0.9826 - val_loss: 0.0426 - val_accuracy: 0.9824 - val_precision: 0.9492 - val_recall: 0.9970 - lr: 0.0010
Epoch 3/50
5350/5350 [==============================] - ETA: 0s - loss: 0.0352 - accuracy: 0.9865 - precision: 0.9676 - recall: 0.9879
Epoch 3: val_loss improved from 0.04259 to 0.03245, saving model to best_lstm_model.h5
5350/5350 [==============================] - 336s 63ms/step - loss: 0.0352 - accuracy: 0.9865 - precision: 0.9676 - recall: 0.9879 - val_loss: 0.0325 - val_accuracy: 0.9837 - val_precision: 0.9525 - val_recall: 0.9978 - lr: 0.0010
Epoch 4/50
5350/5350 [==============================] - ETA: 0s - loss: 0.0279 - accuracy: 0.9894 - preci

In [62]:
!python train.py --img 640 --batch 16 --epochs 30 --data data/ice_texi.yaml --weights yolov5s.pt --name ice_texi_mouth --workers 8


2025-05-07 19:46:22.224756: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-07 19:46:22.224785: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-07 19:46:22.225354: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
train: weights=yolov5s.pt, cfg=, data=data/ice_texi.yaml, hyp=data/hyps/hyp.scratch-low.yaml, epochs=30, batch_size=16, imgsz=640, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, evolve_population=data/hyps, resume_evolve=None, bucket=, cache=None, image_weights=False, device=, multi_scale=False, single_cls=Fal

In [17]:
pip install opencv-python


Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 87.0 MB/s eta 0:00:0000:0100:01m
Note: you may need to restart the kernel to use updated packages.


In [58]:
!python detect.py \
  --weights runs/train/ice_eye_mouth20/weights/best.pt \
  --source data/all_images_준통제 \
  --img 640 \
  --conf-thres 0.65 \
  --save-txt \
  --save-conf \
  --project runs/detect \
  --name ice_yolo_test20_준통제 \
  --exist-ok


detect: weights=['runs/train/ice_eye_mouth20/weights/best.pt'], source=data/all_images_준통제, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.65, iou_thres=0.45, max_det=1000, device=, view_img=False, save_txt=True, save_conf=True, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=ice_yolo_test20_준통제, exist_ok=True, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
WARNING ⚠️ requirements: /opt/conda/lib/python3.11/site-packages/requirements.txt not found, check failed.
YOLOv5 🚀 v7.0-417-g9e6d7c1b Python-3.11.9 torch-2.2.2+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 12010MiB)

Fusing layers... 
Model summary: 157 layers, 7026307 parameters, 0 gradients, 15.8 GFLOPs
image 1/5449 /home/jovyan/ice_sleep_detpj/data/all_images_준통제/Q_098_20_F_01_M0_G0_C0_01.jpg: 640x384 1 Leye_Closed, 1 Reye_Closed, 1 Mouth_Closed, 36.7ms
image 2/5449 /home/jovyan/ice_sleep_detpj/